In [30]:
#  Get input data for time of scan
import os
import re
from datetime import datetime, timedelta
import scipy.io
import meme.archive

In [29]:
inputs = [
    "CAMR:IN20:186:XRMS",
    "CAMR:IN20:186:YRMS",
    "SOLN:IN20:121:BCTRL",
    "QUAD:IN20:121:BCTRL",
    "QUAD:IN20:122:BCTRL",
    "ACCL:IN20:300:L0A_PDES",
    "ACCL:IN20:400:L0B_PDES",
    "QUAD:IN20:361:BCTRL",
    "QUAD:IN20:371:BCTRL",
    "QUAD:IN20:425:BCTRL",
    "QUAD:IN20:441:BCTRL",
    "QUAD:IN20:511:BCTRL",
    "QUAD:IN20:525:BCTRL"
]

In [31]:
scans_dir = f"matlab_scans/"
regex = r"^Emittance-scan-OTRS_IN20_571-202[0-9]-[0-9]{2}-[0-9]{2}-[0-9]{6}\.mat$"


filenames=[]
for file in os.scandir(scans_dir):
    if re.match(regex, file.name):
        filenames.append(file.name)
print('Total ',len(filenames),'files')

Total  51 files


In [32]:
fname = 'Emittance-scan-OTRS_IN20_571-2021-05-01-164241.mat'

def matlab_datenum_to_datetime(datenum):
    datenum = float(datenum)
    return datetime.fromordinal(int(datenum)) + timedelta(days=datenum % 1) - timedelta(days=366)

mat_data = scipy.io.loadmat('matlab_scans/' + fname)['data'][0]
ts = mat_data['ts'][0][0].item()  
ts = matlab_datenum_to_datetime(ts).strftime('%Y-%m-%d %H:%M:%S')

In [33]:
d_files = {f: {} for f in filenames}

for fname in filenames:
    mat_data = scipy.io.loadmat('matlab_scans/' + fname)['data'][0]
    ts = mat_data['ts'][0][0].item()  
    ts = matlab_datenum_to_datetime(ts).strftime('%Y-%m-%d %H:%M:%S')
    results = meme.archive.get(inputs, from_time=ts, to_time=ts)
    
    for pv in results:
        d_files[fname][pv["pvName"]] = (pv["value"]["value"]["values"].item())

    print(d_files)

{'Emittance-scan-OTRS_IN20_571-2021-02-18-125147.mat': {'CAMR:IN20:186:XRMS': 299.77038296884723, 'CAMR:IN20:186:YRMS': 286.9093895129281, 'SOLN:IN20:121:BCTRL': 0.466354, 'QUAD:IN20:121:BCTRL': 0.0029319644975146636, 'QUAD:IN20:122:BCTRL': -0.001687306762141537, 'ACCL:IN20:300:L0A_PDES': 0.0, 'ACCL:IN20:400:L0B_PDES': -2.5, 'QUAD:IN20:361:BCTRL': -3.4654462023910315, 'QUAD:IN20:371:BCTRL': 2.622527052071141, 'QUAD:IN20:425:BCTRL': -2.389879038250811, 'QUAD:IN20:441:BCTRL': 0.22964225678150957, 'QUAD:IN20:511:BCTRL': 3.8276141057810618, 'QUAD:IN20:525:BCTRL': -1.9994971579661676}, 'Emittance-scan-OTRS_IN20_571-2021-05-01-114946.mat': {}, 'Emittance-scan-OTRS_IN20_571-2020-06-21-165108.mat': {}, 'Emittance-scan-OTRS_IN20_571-2021-02-11-193249.mat': {}, 'Emittance-scan-OTRS_IN20_571-2021-06-08-201153.mat': {}, 'Emittance-scan-OTRS_IN20_571-2020-06-21-083923.mat': {}, 'Emittance-scan-OTRS_IN20_571-2021-05-01-115722.mat': {}, 'Emittance-scan-OTRS_IN20_571-2020-06-21-145152.mat': {}, 'Emitt

In [34]:
import json

with open("pv_data.json", "w") as f:
    json.dump(d_files, f, indent=4)